# 손실함수

## 1. Binary Cross-Entrophy, BCE ( 이진 교차 엔트로피 )

- 이진 분류 문제에서 사용되는 손실 함수 
- 모델이 예측한 확률과 실제 레이블 사이의 차이를 직접적으로 계산, 성능 개선에 도움을 준다.  

### 단점

- 이상치에 민감, 특히 예측 확률이 0또는 1에 가까울때 민감
- 이진 분류에 만 한정 




# BCE 이진 교차 엔트로피: 수식, 역사, 유도
BCE(Binary Cross-Entropy)는 이진분류에서 예측 확률과 정답 분포의 차이를 측정하는 대표 손실함수입니다.

### 1) 수식
정답 $y \in \{0,1\}$, 모델의 클래스 1 예측확률을 $p$라고 하면 샘플 1개의 BCE는:

$$
\mathcal{L}_{BCE}(y,p)= -\big(y\log p + (1-y)\log(1-p)\big)
$$

데이터 $N$개 평균 손실:

$$
\mathcal{L}= -\frac{1}{N}\sum_{i=1}^{N}\Big(y_i\log p_i+(1-y_i)\log(1-p_i)\Big)
$$

### 2) 역사(핵심 흐름)
- 1948년: Shannon의 정보이론에서 엔트로피 개념 정립
- 1950년대: 통계학의 Bernoulli/로지스틱 회귀에서 최대우도추정(MLE) 사용
- 이후: 신경망 분류 학습에서 음의 로그우도(NLL) 형태가 BCE로 표준화
- 현대 딥러닝: 수치 안정성을 위해 BCEWithLogitsLoss(로짓 직접 입력) 형태를 많이 사용

### 3) 식의 유도
Bernoulli 분포 가정에서 시작합니다.

한 샘플의 우도:
$$
P(y|x)=p^y(1-p)^{1-y}
$$

데이터 전체 우도:
$$
\prod_{i=1}^N p_i^{y_i}(1-p_i)^{1-y_i}
$$

로그우도:
$$
\log \mathcal{L}_{lik}=\sum_{i=1}^N\Big(y_i\log p_i+(1-y_i)\log(1-p_i)\Big)
$$

학습은 보통 “최대 로그우도”와 동치인 “음의 로그우도 최소화”를 하므로:

$$
-\frac{1}{N}\log \mathcal{L}_{lik}
=
-\frac{1}{N}\sum_{i=1}^N\Big(y_i\log p_i+(1-y_i)\log(1-p_i)\Big)
$$

이 식이 바로 BCE입니다.

직관:
- 정답이 1인데 $p \to 0$이면 손실이 매우 커짐
- 정답이 0인데 $p \to 1$이어도 손실이 매우 커짐
- 맞는 쪽 확률을 높일수록 손실이 작아짐

### 추가 자료
- [Cross entropy (Wikipedia)](https://en.wikipedia.org/wiki/Cross_entropy)
- [PyTorch BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

In [ ]:
import torch
import torch.nn as nn

y_pred = torch.tensor([0.2, 0.8, 0.0], dtype=torch.float)
y_true = torch.tensor([0, 1, 0], dtype=torch.float)

bce_loss = nn.BCELoss()
y_pred_sigmoid = torch.sigmoid(y_pred) #예측값에 시그모이드 활성화 함수를 적용 
# 이진 분류 문제에서 모델의 출력을 확률로 해석하기 위해 필요한 단계 ( 0과 1사이의 확률 값으로 반환 ) 
loss = bce_loss(y_pred_sigmoid, y_true)

print(f'Binary Cross-Entropy Loss: {loss.item()}')

## 다중 분류 교차 엔트로피 손실(Categorical Cross-Entropy Loss)

- 세개 이상의 클래스를 가진 분류 문제에서 사용되는 손실 함수 
- 다중 클래스 분류 문제에서 모델의 성능을 정확하게 평가 할 수 있다. 
- 모델이 각 클래스에 속할 확률을 학습하도록 하여, 예측의 불확실 성을 줄일 수 있다. 

### 단점

- 레이블이 one-hot 인코딩 되어야 하며, 클래스가 많을 수록 계산 비용이 증가 
- 불균형한 데이터 셋에서는 특정 클래스에 대한 모델의 성능이 과대평가되거나 과소평가될 수 있음.  

Optimized tool selection# 다중 분류 엔트로피 손실, 유도, 역사, One-hot 인코딩
다중 분류에서 가장 널리 쓰는 손실은 교차 엔트로피 손실이며, 정답 클래스 확률을 크게 만들도록 학습시키는 음의 로그우도와 같은 형태입니다.


$$
L = -\sum_{k=1}^{K} y_k \log p_k
$$

- $K$: 클래스 개수  
- $y_k$: 정답 레이블(One-hot이면 정답 클래스만 1, 나머지 0)  
- $p_k$: 모델이 예측한 클래스 $k$의 확률(보통 softmax 출력)

데이터셋 전체에서는 평균을 씁니다.

$$
J = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K} y_{ik}\log p_{ik}
$$

유도 과정 핵심:
1. 다중 분류에서 한 샘플의 정답 레이블 분포를 범주분포로 둡니다.  
2. One-hot 레이블이면 우도는
$$
P(y|x)=\prod_{k=1}^{K} p_k^{y_k}
$$
3. 로그를 취하면
$$
\log P(y|x)=\sum_{k=1}^{K} y_k \log p_k
$$
4. 최대우도추정(MLE)은 로그우도 최대화, 즉 음의 로그우도 최소화와 같아서
$$
L=-\sum_{k=1}^{K} y_k\log p_k
$$
가 됩니다.  
5. One-hot 특성 때문에 실제로는 정답 클래스 $c$에 대해 $L=-\log p_c$와 동일합니다.

역사 요약:
1. 1948년: Shannon이 엔트로피 개념을 정보이론에 정립.  
2. 1951년: Kullback-Leibler가 분포 차이를 정량화(KL divergence).  
3. 이후 통계학의 최대우도·로그손실과 결합되어 분류 손실로 자리잡음.  
4. 1980년대 이후 신경망+역전파 확산, 2010년대 딥러닝 표준 손실로 사실상 정착.

One-hot 인코딩 간단 예시:
- 클래스 집합: 고양이, 강아지, 새
- 인코딩:
1. 고양이 → $[1,0,0]$
2. 강아지 → $[0,1,0]$
3. 새 → $[0,0,1]$

예를 들어 정답이 강아지이고 모델 확률이 $[0.1,0.7,0.2]$이면 손실은
$$
L = -(0\log0.1 + 1\log0.7 + 0\log0.2) = -\log0.7
$$
이라서, 정답 확률 $0.7$이 더 커질수록 손실이 작아집니다.

### 추가 자료
- [Cross entropy 설명(위키피디아)](https://en.wikipedia.org/wiki/Cross_entropy)
- [Softmax 함수 설명](https://en.wikipedia.org/wiki/Softmax_function)

- 다중 분류 교차 엔트로피 손실 함수 CCE를 사용 하는 경우, 예측확률과 실제 레이블과 얼마나 잘 일치하는지를 어떻게 측정?
    - 실제 레이블이 1인 경우에만 $y_{i}\log{\hat{y}_{i}}$

### 📊 딥러닝 층별 핵심 함수 한눈에 보기

| 단계 (Layer) | 주요 역할 | 주로 사용되는 함수 | 목적 |
| --- | --- | --- | --- |
| **1. 입력층** | 원본 데이터 수용 및 정제 | (엄밀한 활성화 함수는 없음)<br>

<br>정규화/표준화 함수 | 데이터의 스케일을 맞춰 학습 속도 향상 |
| **2. 은닉층** | 데이터의 특징 추출 및 비선형성 부여 | **ReLU**, Leaky ReLU, tanh 등 | 기울기 소실 방지, 복잡한 패턴 학습 |
| **3. 출력층** | 최종 결과(예측값) 도출 | **선형(항등) 함수, Sigmoid, Softmax** | 문제의 종류(회귀, 분류)에 맞는 형태 출력 |
| **4. 역전파** | 오차 계산 및 가중치 업데이트 | **손실 함수(MSE, Cross-Entropy)의 미분**<br>

<br>각 층 **활성화 함수의 미분** | 오차를 줄이는 방향으로 모델 교정 |

---

### 1. 입력층 (Input Layer): "데이터의 출입구"

입력층은 외부의 데이터(이미지 픽셀, 텍스트 벡터 등)를 신경망 안으로 들여보내는 역할을 합니다.

* **사용 함수:** 입력층 자체에는 퍼셉트론의 내적이나 활성화 함수가 들어가지 않습니다. 대신 데이터를 모델이 소화하기 좋게 다듬는 전처리 과정의 함수들이 쓰입니다.
* **대표 예시:** * **정규화(Normalization):** 데이터의 범위를 0~1 사이로 맞춥니다.
* **표준화(Standardization):** 평균을 0, 분산을 1로 맞춥니다. (예: $z = \frac{x - \mu}{\sigma}$)


* **이유:** 키(170cm)와 시력(1.2)처럼 단위와 크기가 전혀 다른 데이터가 그대로 들어오면 모델이 혼란스러워하기 때문에, 출발선을 맞춰주는 작업입니다.

### 2. 은닉층 (Hidden Layer): "핵심 두뇌 발전소"

입력된 데이터에서 복잡한 특징과 패턴을 찾아내는 진짜 딥러닝이 일어나는 공간입니다. 수백, 수천 개의 층이 겹겹이 쌓여 있습니다.

* **사용 함수:** **활성화 함수 (Activation Function)**
* **대표 예시:** * **ReLU:** $f(x) = \max(0, x)$. 은닉층의 절대 강자입니다. 계산이 엄청 빠르고, 신경망이 깊어져도 학습이 잘 됩니다(기울기 소실 해결).
* **Leaky ReLU:** ReLU의 단점(0 이하에서 뉴런이 죽어버리는 현상)을 보완하여 0 이하에서도 미세한 기울기를 살려둡니다.
* **tanh:** 결과값을 -1에서 1 사이로 맞춰주며, 순환 신경망(RNN) 등 특정 구조의 은닉층에서 여전히 자주 쓰입니다.


* **이유:** 은닉층에 이 활성화 함수(비선형 함수)들이 없다면, 아무리 층을 깊게 쌓아도 결국 거대한 선형 회귀 1개를 푼 것과 똑같아져 복잡한 문제를 풀 수 없습니다.

### 3. 출력층 (Output Layer): "최종 결론 창구"

은닉층이 열심히 분석한 결과를 바탕으로, 우리가 원하는 최종 답안지 형식으로 변환하여 출력합니다. **풀고자 하는 문제의 종류에 따라 사용하는 함수가 완전히 달라집니다.**

* **대표 예시:**
* **항등 함수 (Identity Function):** $f(x) = x$. 들어온 값을 그대로 출력합니다. 아파트 가격, 내일의 온도 등 연속적인 수치(회귀 문제)를 예측할 때 씁니다.
* **시그모이드 (Sigmoid):** 출력값을 0~1 사이의 확률로 압축합니다. 고양이인지 강아지인지, 합격인지 불합격인지 판단하는 **이진 분류 문제**에 쓰입니다.
* **소프트맥스 (Softmax):** 여러 개의 출력값을 각각 0~1 사이의 확률로 만들고, 그 합을 무조건 1로 맞춰줍니다. 가위/바위/보 중 무엇인지, 숫자 0~9 중 무엇인지 맞히는 **다중 분류 문제**에 쓰입니다.



### 4. 역전파 (Backpropagation): "오답 노트와 교정"

출력층에서 나온 결과가 정답과 얼마나 다른지 확인하고, 거꾸로 돌아가며 신경망의 가중치를 수정하는 학습 과정입니다.

* **사용 함수:** 이곳에서는 모델을 구성하는 함수가 아니라, '오차를 평가하고 수정하기 위한 수학적 도구'로서의 함수들이 쓰입니다.
* **대표 예시:**
* **손실 함수 (Loss Function):** 오차를 숫자로 계산하는 함수입니다. 회귀에는 주로 **MSE (평균 제곱 오차)**, 분류에는 주로 Cross-Entropy (교차 엔트로피)를 사용합니다.
* **미분 함수 (Derivatives):** 역전파는 "기울기(미분값)"를 타고 내려가는 과정입니다. 따라서 출력층부터 은닉층까지 사용되었던 모든 함수(Softmax, ReLU 등)의 **미분 함수**가 연쇄 법칙(Chain Rule)에 의해 쫙 곱해지며 사용됩니다.
* **최적화 함수 (Optimizer):** 미분하여 구한 기울기를 바탕으로 가중치를 '얼마나, 어떤 방식으로' 업데이트할지 결정하는 함수입니다. **Adam, SGD, RMSprop** 등이 대표적이며, 오차의 바닥을 향해 가장 빠르고 안정적으로 내려가는 길잡이 역할을 합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 실제 레이블과 예측값 정의
# 실제 레이블은 클래스 인덱스로 제공됩니다.
y_true = torch.tensor([1, 0, 2])  # 클래스 인덱스 형태의 실제 레이블

# 모델 예측값 (로짓 값, 즉 softmax를 적용하기 전의 값)
y_pred_logits = torch.tensor([[2.0, 1.0, 0.1], 
                              [0.1, 1.5, 1.2], 
                              [1.0, 0.2, 3.0]])

# CrossEntropyLoss는 내부적으로 softmax를 적용하므로, 로짓 값을 직접 넘깁니다.
cross_entropy_loss = nn.CrossEntropyLoss()# 손실 함수 객체 생성 

# 손실 계산
loss = cross_entropy_loss(y_pred_logits, y_true)

print(f'Categorical Cross-Entropy Loss: {loss.item()}')
print("손실값이 낮을 수록 모델의 예측이 정확하다는 것, 높은 손실 값은 예측의 정확성이 낮음을 의미")